# 03 — Gradient boosting for classification

*Chapter 08 · Gradient Boosting · Notebook 3 of 6*

Notebook 2 left us with a recipe: gradient boosting fits the **negative gradient of whatever loss you
choose**. We have only used it for regression. Now we change the loss — and the very same machine starts
doing **classification**. Along the way we meet the one honest extra step classification needs (a Newton
update in each leaf), and we close a loop opened in Chapter 07: **AdaBoost turns out to be gradient
boosting with one particular loss.**

**Prerequisites.** Notebook 1 (the residual-fitting loop); Notebook 2 (the residual is the negative
gradient; fit the negative gradient of any loss); Chapter 03 (the sigmoid, log-odds, and log-loss);
Chapter 07 (AdaBoost).

**What you'll be able to do.**
- Do gradient-boosting classification by hand: log-loss gives the pseudo-residual `y − p`.
- Apply the honest **Newton leaf-step** and match `GradientBoostingClassifier` exactly.
- See why the naive regression-mean leaf gives a different model.
- See that **AdaBoost is the exponential-loss member** of the gradient-boosting family.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()
SEED, NU, MAX_DEPTH, N_TREES = 0, 0.3, 2, 50


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


# Chapter 07's two-moons set: the same problem AdaBoost learned, so we can compare.
X, y_all = make_moons(n_samples=400, noise=0.20, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_all, test_size=0.30, random_state=SEED, stratify=y_all
)
print(
    f"train {X_train.shape[0]} points, test {X_test.shape[0]};  "
    f"base rate p0 = {y_train.mean():.3f}"
)

## A recap, and the plan

Notebook 2's lesson: gradient boosting fits the **negative gradient of whatever loss you choose**, and a
different loss gives a different thing to fit. For classification, the right loss is **log-loss**.

**Re-lay Chapter 03.** A classifier produces a real-valued score `F(x)` — the **log-odds** — and squashes
it to a probability with the sigmoid, `p = σ(F) = 1 / (1 + e^{−F})`. **Log-loss** (cross-entropy) measures
how good that probability is, punishing confident-and-wrong predictions without bound. Its negative
gradient with respect to the score is

$$-\frac{\partial L}{\partial F} = y - p,$$

the gap between the label and the predicted probability — "how wrong we are right now." That is the
pseudo-residual we will fit.

## Boost in log-odds space

We build the score `F(x)` additively, exactly as before, but now it lives in log-odds space and we read
it through the sigmoid.

- **Start** at the best constant score: `F₀ = log-odds of the base rate`. Our two-moons set is balanced
  (half each class), so `F₀ = log(0.5 / 0.5) = 0`, and every point starts at `p = 0.5`.
- **Each round:** form the pseudo-residual `y − p` (positive for a class-1 point — push its score up;
  negative for class-0 — push it down), fit a **regression** tree to it, add a shrunken slice.

The target `y` is a class, but the thing we fit each round is the continuous `y − p`. The regression
machinery of Notebooks 1–2 carries over unchanged — almost.

In [ ]:
# Start from the best constant score: the log-odds of the base rate.
p0 = y_train.mean()
F0 = np.log(p0 / (1 - p0))
p = sigmoid(np.full(len(y_train), F0))
r = y_train - p  # the pseudo-residual: the negative gradient of log-loss

print(f"F0 = log-odds({p0:.3f}) = {F0:.4f}   ->   p = sigmoid(F0) = {p[0]:.3f}")
print(f"round-1 pseudo-residuals  y - p  take values: {np.unique(np.round(r, 3))}")
tree1 = DecisionTreeRegressor(max_depth=MAX_DEPTH, random_state=SEED).fit(X_train, r)

**Read the result.** Because every point starts at `p = 0.5`, the round-1 pseudo-residual `y − p`
is `+0.5` for class-1 points and `−0.5` for class-0 — a clean "push up / push down" signal in score
space. We fit a regression tree to it, as in Notebook 1. The only thing that changes for classification
is what value each leaf should hold.

## The leaf value: an honest Newton step

In Notebook 1, a regression tree put the **mean** of a leaf's residuals — and for squared error that was
exactly the optimal step. For **log-loss it is not optimal.** Log-loss curves: its second derivative at a
point is `p(1 − p)`, the variance of a Bernoulli outcome with probability `p`. The best value to add in a
leaf accounts for that
curvature — it is the one-step **Newton update**

$$\gamma = \frac{\sum_{\text{leaf}} (y - p)}{\sum_{\text{leaf}} p(1 - p)}.$$

Since `p(1 − p) ≤ 0.25`, the denominator is small and `γ` is *larger* than the plain mean — a
better-scaled step toward the minimum. scikit-learn applies this update inside every leaf for you. If we
keep the naive mean, we build a different (and slower) model. Let us do the honest version, and check
it.

In [ ]:
def boost_classify(X, y, n_trees, lr, depth, leaf):
    """By-hand log-loss gradient boosting. leaf in {'newton', 'mean'}."""
    F0 = np.log(y.mean() / (1 - y.mean()))
    F = np.full(len(y), F0)
    trees, leaf_values = [], []
    for _ in range(n_trees):
        p = sigmoid(F)
        r = y - p
        tree = DecisionTreeRegressor(max_depth=depth, random_state=SEED).fit(X, r)
        leaves = tree.apply(X)
        gamma = {}
        for leaf_id in np.unique(leaves):
            mask = leaves == leaf_id
            if leaf == "newton":
                gamma[leaf_id] = r[mask].sum() / (p[mask] * (1 - p[mask])).sum()
            else:  # the naive regression mean
                gamma[leaf_id] = r[mask].mean()
        F = F + lr * np.array([gamma[i] for i in leaves])
        trees.append(tree)
        leaf_values.append(gamma)
    return F0, F, trees, leaf_values


def score(X_query, F0, trees, leaf_values, lr):
    F = np.full(len(X_query), F0)
    for tree, gamma in zip(trees, leaf_values, strict=False):
        leaves = tree.apply(X_query)
        F = F + lr * np.array([gamma.get(i, 0.0) for i in leaves])
    return F


# By hand, with the Newton leaf, and the same settings as the library.
F0, F_newton, trees, vals = boost_classify(X_train, y_train, N_TREES, NU, MAX_DEPTH, "newton")
gbc = GradientBoostingClassifier(
    loss="log_loss", n_estimators=N_TREES, learning_rate=NU, max_depth=MAX_DEPTH,
    subsample=1.0, random_state=SEED,
).fit(X_train, y_train)
parity = np.max(np.abs(F_newton - gbc.decision_function(X_train).ravel()))

# The naive mean-leaf variant — a different model.
_, F_mean, _, _ = boost_classify(X_train, y_train, N_TREES, NU, MAX_DEPTH, "mean")
ll_newton = log_loss(y_train, sigmoid(F_newton))
ll_mean = log_loss(y_train, sigmoid(F_mean))

F_test = score(X_test, F0, trees, vals, NU)
acc_test = accuracy_score(y_test, (F_test > 0).astype(int))
print(f"by-hand Newton vs GradientBoostingClassifier:  max|Δ decision_function| = {parity:.2e}")
print(f"train log-loss:  Newton {ll_newton:.4f}   vs   naive mean-leaf {ll_mean:.4f}")
print(f"by-hand test accuracy = {acc_test:.4f}")

**Read the result.** The by-hand Newton model matches `GradientBoostingClassifier` to about
`1e-15` — machine precision. So this *is* the library's algorithm: log-loss pseudo-residual, regression
tree, Newton leaf, shrunken step. The naive **mean-leaf** model is a different one — here its training
log-loss is several times higher, because it under-steps (it ignores the curvature `p(1 − p)`). The exact
gap depends on the settings, but the pattern is robust: the Newton leaf matches the library and beats the
mean. This Newton step is the single place where classification needs more than Notebook 1's recipe — and
because scikit-learn does it silently, we did it in the open.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, n in zip(axes, (1, 10, 50), strict=False):
    clf = GradientBoostingClassifier(
        loss="log_loss", n_estimators=n, learning_rate=NU, max_depth=MAX_DEPTH, random_state=SEED
    ).fit(X_train, y_train)
    viz.plot_decision_boundary(clf, X_train, y_train, ax=ax)
    ax.set_title(f"{n} tree{'s' if n > 1 else ''}   (test acc {clf.score(X_test, y_test):.3f})")
plt.show()

**Read the figure.** With a single tree the boundary is one coarse split. By ten trees it bends
around the two crescents; by fifty it traces them cleanly (test accuracy ≈ 0.94). As trees accumulate the
score `F` sharpens and the predicted probabilities pull toward 0 and 1 — the same build-up as the
regression fit in Notebook 1, now seen as a hardening decision boundary.

In [ ]:
# Train log-loss after each tree: by-hand Newton, by-hand mean-leaf, and the library.
def staged_logloss(X, y, n_trees, lr, depth, leaf):
    F0 = np.log(y.mean() / (1 - y.mean()))
    F = np.full(len(y), F0)
    curve = []
    for _ in range(n_trees):
        p = sigmoid(F)
        r = y - p
        tree = DecisionTreeRegressor(max_depth=depth, random_state=SEED).fit(X, r)
        leaves = tree.apply(X)
        gamma = {}
        for leaf_id in np.unique(leaves):
            mask = leaves == leaf_id
            if leaf == "newton":
                gamma[leaf_id] = r[mask].sum() / (p[mask] * (1 - p[mask])).sum()
            else:
                gamma[leaf_id] = r[mask].mean()
        F = F + lr * np.array([gamma[i] for i in leaves])
        curve.append(log_loss(y, sigmoid(F)))
    return np.array(curve)


ll_newton_curve = staged_logloss(X_train, y_train, N_TREES, NU, MAX_DEPTH, "newton")
ll_mean_curve = staged_logloss(X_train, y_train, N_TREES, NU, MAX_DEPTH, "mean")
ll_sklearn_curve = np.array(
    [log_loss(y_train, proba) for proba in gbc.staged_predict_proba(X_train)]
)

rounds = np.arange(1, N_TREES + 1)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rounds, ll_newton_curve, color=COLORS["model"], linewidth=2.6, label="by hand: Newton leaf")
ax.plot(rounds, ll_sklearn_curve, color=COLORS["highlight"], linewidth=1.4, linestyle="--",
        label="GradientBoostingClassifier")
ax.plot(rounds, ll_mean_curve, color=COLORS["error"], linewidth=2.2,
        label="by hand: naive mean leaf")
ax.set_xlabel("number of trees")
ax.set_ylabel("train log-loss")
ax.set_title("the Newton leaf matches the library; the mean leaf lags")
ax.legend()
plt.show()

**Read the figure.** The by-hand Newton curve drops quickly and sits *exactly* on scikit-learn's
(the dashed line) — visual confirmation of the machine-precision match. The naive mean-leaf curve stays
above it: a different, slower model that would need many more trees to reach the same fit, because its
leaf steps are mis-scaled. The leaf value is not a footnote — it is the difference between the library's
model and another one.

## The unifying reveal: where AdaBoost lives

We chose log-loss. Chapter 07's AdaBoost minimized a different loss — the **exponential** loss
`e^{−margin}`. By Notebook 2's logic, that is another loss to plug into the same machine. So AdaBoost
ought to be gradient boosting with `loss='exponential'`. Let us check, on the very same two-moons set
AdaBoost learned in Chapter 07.

In [ ]:
exp_gb = GradientBoostingClassifier(
    loss="exponential", n_estimators=N_TREES, learning_rate=1.0, max_depth=1, random_state=SEED
).fit(X_train, y_train)
ada = AdaBoostClassifier(
    n_estimators=N_TREES, learning_rate=1.0, random_state=SEED
).fit(X_train, y_train)

agree_train = (exp_gb.predict(X_train) == ada.predict(X_train)).mean()
agree_test = (exp_gb.predict(X_test) == ada.predict(X_test)).mean()
print(f"exponential-loss GB  test acc = {exp_gb.score(X_test, y_test):.4f}")
print(f"AdaBoost             test acc = {ada.score(X_test, y_test):.4f}")
print(f"prediction agreement:  train {agree_train:.3f}   test {agree_test:.3f}")

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 5))
viz.plot_decision_boundary(exp_gb, X_train, y_train, ax=axL)
axL.set_title(f"GB, loss='exponential'   (test acc {exp_gb.score(X_test, y_test):.3f})")
viz.plot_decision_boundary(ada, X_train, y_train, ax=axR)
axR.set_title(f"AdaBoost   (test acc {ada.score(X_test, y_test):.3f})")
plt.show()

**Read the figure.** The two decision boundaries are nearly the same, and the test accuracies are
identical — yet the models agree on about 98% of points, **not 100%**. They share the **objective** (the
exponential loss) but run **different optimizers**: gradient boosting takes functional-gradient steps with
Newton leaves and shrinkage, while AdaBoost reweights the points and votes with `α = ln((1 − ε)/ε)`. Same
destination, different road. So AdaBoost is the **exponential-loss member** of the gradient-boosting
family — the bridge promised at the end of Chapter 07, now crossed. Gradient boosting is the general
method; AdaBoost is one choice of loss.

## Your turn

1. **Why the mean lags.** From the log-loss figure, explain in one sentence why the naive mean-leaf curve
   sits above the Newton curve — what does the mean ignore that the Newton step uses?
2. **Closer with more rounds.** Re-run the exponential-GB vs AdaBoost comparison with `N_TREES = 100`.
   Does the prediction agreement rise or fall? What does that suggest about whether they converge to the
   same model?
3. **Why the Newton step is bigger.** Show that the Newton leaf `Σ(y − p) / Σ p(1 − p)` is always at least
   as large in magnitude as the plain mean `Σ(y − p) / count`, using `p(1 − p) ≤ 0.25` (and that
   `0.25 < 1`). Explain what that means for the step size, and why log-loss needs it where squared error
   did not.

## What you built

- You did gradient-boosting **classification** by hand: log-loss gives the pseudo-residual `y − p`, and
  you fit a regression tree to it in log-odds space.
- You applied the honest **Newton leaf-step** `Σ(y − p) / Σ p(1 − p)` and matched
  `GradientBoostingClassifier` to machine precision — and saw that the naive mean leaf is a different
  model.
- You showed that **AdaBoost is the exponential-loss member** of the gradient-boosting family, closing the
  bridge from Chapter 07.

**Vocabulary you now own:** log-odds score · log-loss · pseudo-residual `y − p` · Newton leaf-step · loss
curvature `p(1 − p)` · exponential-loss member.

## Going further (optional)

- **Where the Newton leaf comes from.** Inside a leaf, we want the constant step `γ` that most reduces the
  log-loss of the points there. A one-step Newton minimization (a second-order Taylor expansion of the
  loss in `γ`) gives `γ = Σ(y − p) / Σ p(1 − p)`: the numerator is the gradient, the denominator the
  curvature `p(1 − p)`. This is Friedman's TreeBoost update (2001, §4.5–4.6). Notebook 2 called the
  per-leaf value a *line search*; for log-loss that line search is a Newton step, and for squared error
  (Notebook 1) it collapsed to the mean.
- **More than two classes.** Multiclass gradient boosting fits **one regression tree per class** each
  round and combines them through a softmax — the same idea, K scores instead of one.

## References

- Friedman, J. H. (2001). *Greedy function approximation: a gradient boosting machine.* Annals of
  Statistics, 29(5), 1189–1232. DOI [10.1214/aos/1013203451](https://doi.org/10.1214/aos/1013203451).
  (§4.5–4.6, the classification / TreeBoost leaf update.)
- Friedman, J., Hastie, T., & Tibshirani, R. (2000). *Additive logistic regression: a statistical view of
  boosting.* Annals of Statistics, 28(2), 337–407. DOI
  [10.1214/aos/1016218223](https://doi.org/10.1214/aos/1016218223). (AdaBoost = forward-stagewise
  exponential loss.)
- The sigmoid, log-odds, and log-loss: Chapter 03. AdaBoost: Chapter 07.

*Previous: 02 — The residual was the gradient.*  ·  *Next: 04 — Shrinkage and the trees.*